# Challenge 9 - Análisis de Actividad de Producto
### 1️⃣ Exploración Inicial (Medir antes de limpiar)

In [1]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset
df_raw = pd.read_csv('docs/product_activity.csv')

# Mostrar exploracion basica
print("--- head() ---")
display(df_raw.head())
print("\n--- info() ---")
df_raw.info()
print("\n--- describe() ---")
display(df_raw.describe())

# Conteo de nulos y duplicados exactos
print("\n--- Nulos por columna ---")
print(df_raw.isnull().sum())
print(f"\nDuplicados exactos: {df_raw.duplicated().sum()}")

# Valores unicos y frecuencias
print("\n--- plan_type ---")
print(df_raw['plan_type'].value_counts(dropna=False))
print("\n--- post_category ---")
print(df_raw['post_category'].value_counts(dropna=False))
print("\n--- device_type ---")
print(df_raw['device_type'].value_counts(dropna=False))

# Chequeos lógicos: Posts antes del signup
temp_signup = pd.to_datetime(df_raw['created_at'], errors='coerce')
temp_post = pd.to_datetime(df_raw['post_created_at'], errors='coerce')
posts_before_signup = (temp_post < temp_signup).sum()
print(f"\nPosts que ocurren antes del signup: {posts_before_signup}")

--- head() ---


,user_id,created_at,country,plan_type,user_age,post_id,post_category,post_created_at,votes_received,user_total_posts,days_since_signup,device_type
0,U01988,2025-02-18T02:07:44,PY,pro,26.0,P0008515,sport,2025-05-07T20:55:28,7,16,78,mobile
1,U00236,2025-06-22T07:49:10,BR,free,27.0,P0001023,tech,2025-09-13T20:31:06,1,9,83,web
2,U00791,2024-02-12T02:45:45,CL,free,28.0,P0003405,tech,2024-02-14T05:17:48,11,2,2,mobile
3,U01522,2024-09-22T07:06:50,US,free,16.0,P0006524,finance,2024-09-24T07:51:34,5,2,2,web
4,U01092,2025-07-18T02:27:52,PY,free,NaN,P0004665,education,2025-07-24T04:56:56,7,2,6,mobile



--- info() ---
<class 'pandas.DataFrame'>
RangeIndex: 8782 entries, 0 to 8781
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            8782 non-null   str    
 1   created_at         8782 non-null   str    
 2   country            8782 non-null   str    
 3   plan_type          8782 non-null   str    
 4   user_age           8028 non-null   float64
 5   post_id            8782 non-null   str    
 6   post_category      8782 non-null   str    
 7   post_created_at    8782 non-null   str    
 8   votes_received     8782 non-null   int64  
 9   user_total_posts   8782 non-null   int64  
 10  days_since_signup  8782 non-null   int64  
 11  device_type        8782 non-null   str    
dtypes: float64(1), int64(3), str(8)
memory usage: 823.4 KB

--- describe() ---


,user_age,votes_received,user_total_posts,days_since_signup
count,8028.000000,8782.000000,8782.000000,8782.000000
mean,27.902591,6.918356,8.324186,29.479390
std,7.547052,5.127311,6.754906,36.819928
min,16.000000,0.000000,1.000000,0.000000
25%,22.000000,3.000000,4.000000,5.000000
50%,28.000000,6.000000,6.000000,17.000000
75%,33.000000,9.000000,11.000000,40.000000
max,58.000000,74.000000,39.000000,404.000000



--- Nulos por columna ---
user_id                0
created_at             0
country                0
plan_type              0
user_age             754
post_id                0
post_category          0
post_created_at        0
votes_received         0
user_total_posts       0
days_since_signup      0
device_type            0
dtype: int64

Duplicados exactos: 172

--- plan_type ---
plan_type
free            5978
pro             1460
enterprise       306
Free             208
 free            197
FREE             196
FreE             189
PRo               46
 pro              44
PRO               44
Pro               38
Pro               35
EnterPrise        13
ENTERPRISE        11
 enterprise        7
Enterprise         7
premium            1
vip                1
enterprise+        1
Name: count, dtype: int64

--- post_category ---
post_category
tech           1187
life            913
sports          899
science         753
finance         739
gaming          727
music           614
heal

### 2️⃣ Limpieza Básica con Criterio

In [2]:
df_clean = df_raw.copy()

# Normalizacion Canonica (Diccionarios Fijos)
dic_plan = {'free': 'free', 'pro': 'pro', 'enterprise': 'enterprise'}
dic_device = {'web': 'web', 'mobile': 'mobile', 'desktop': 'desktop'}
dic_category = {
    'tech': 'tech', 'life': 'life', 'sports': 'sports', 'science': 'science', 
    'finance': 'finance', 'gaming': 'gaming', 'music': 'music', 
    'health': 'health', 'education': 'education', 'travel': 'travel'
}

def clean_value(val, valid_dict):
    if pd.isna(val):
        return 'invalid'
    # convertir a str, minuscula y limpiar espacios
    clean_val = str(val).lower().strip()
    return valid_dict.get(clean_val, 'invalid')

df_clean['plan_type'] = df_clean['plan_type'].apply(lambda x: clean_value(x, dic_plan))
df_clean['device_type'] = df_clean['device_type'].apply(lambda x: clean_value(x, dic_device))
df_clean['post_category'] = df_clean['post_category'].apply(lambda x: clean_value(x, dic_category))

# Fechas a datetime
df_clean['created_at'] = pd.to_datetime(df_clean['created_at'], errors='coerce')
df_clean['post_created_at'] = pd.to_datetime(df_clean['post_created_at'], errors='coerce')

# Recalculo obligado de days_since_signup_calc
df_clean['days_since_signup_calc'] = (df_clean['post_created_at'] - df_clean['created_at']).dt.days

# Quarantine: Errores duros
mask_date_error = df_clean['created_at'].isna() | df_clean['post_created_at'].isna()
mask_logic_error = df_clean['days_since_signup_calc'] < 0
mask_dict_error = (df_clean['plan_type'] == 'invalid') | (df_clean['device_type'] == 'invalid') | (df_clean['post_category'] == 'invalid')

mask_quarantine = mask_date_error | mask_logic_error | mask_dict_error

# Separar DataFrames
df_quarantine = df_clean[mask_quarantine].copy()
df_core = df_clean[~mask_quarantine].copy()

# Asignar reason_code al quarantine
def get_quarantine_reason(row):
    if pd.isna(row['created_at']) or pd.isna(row['post_created_at']):
        return 'date_not_parseable'
    if row['days_since_signup_calc'] < 0:
        return 'post_before_signup'
    return 'invalid_dictionary_value'

if not df_quarantine.empty:
    df_quarantine['reason_code'] = df_quarantine.apply(get_quarantine_reason, axis=1)

### 3️⃣ Data Quality Report

In [3]:
total_raw = len(df_raw)
total_core = len(df_core)
pct_quarantine = (len(df_quarantine) / total_raw) * 100 if total_raw > 0 else 0

# Duplicados en CORE
duplicados_core = df_core.duplicated().sum()
df_core = df_core.drop_duplicates()
total_core_dedup = len(df_core)

# Porcentaje de mismatches entre days_since_signup original vs el calculado
mismatches = (df_core['days_since_signup'] != df_core['days_since_signup_calc']).sum()
pct_mismatch = (mismatches / total_core_dedup) * 100 if total_core_dedup > 0 else 0

print(f"Filas RAW iniciales: {total_raw}")
print(f"Filas CORE limpias: {total_core_dedup}")
print(f"% Quirantine (descarte): {pct_quarantine:.2f}%")
print(f"Duplicados limpios en CORE: {duplicados_core}")
print(f"% Mismatch en dias desde signup: {pct_mismatch:.2f}%")

Filas RAW iniciales: 8782
Filas CORE limpias: 8161
% Quirantine (descarte): 5.22%
Duplicados limpios en CORE: 163
% Mismatch en dias desde signup: 50.57%


### 4️⃣ Métricas y Análisis

In [5]:
print("--- Distribuciones ---")
print("\nUsuarios unicos por plan:")
print(df_core.groupby('plan_type')['user_id'].nunique())

print("\nActividad (posts) por pais (Top 5):")
print(df_core['country'].value_counts().head(5))

print("\nActividad (posts) por categoria:")
print(df_core['post_category'].value_counts())

print("\nActividad (posts) por dispositivo:")
print(df_core['device_type'].value_counts())

print("\n--- Engagement ---")
print("\nVotos por plan (media y mediana):")
print(df_core.groupby('plan_type')['votes_received'].agg(['mean', 'median']))

print("\nVotos por categoria (Promedio):")
print(df_core.groupby('post_category')['votes_received'].mean().sort_values(ascending=False))

print("\n--- Evento vs Usuario ---")
promedio_fila = df_core['votes_received'].mean()
promedio_usuario = df_core.groupby('user_id')['votes_received'].mean().mean()
print(f"Promedio global de votos por fila (post): {promedio_fila:.2f}")
print(f"Promedio global de votos por usuario: {promedio_usuario:.2f}")
print("-> Nota: Difieren por que el primer promedio le da mas peso a los usuarios que publican mucho (sesgo por volumen).")

print("\n--- Concentracion y Temporalidad ---")
# Concentracion top 1%
posts_por_usuario = df_core['user_id'].value_counts()
top_1_perc_qty = max(1, int(len(posts_por_usuario) * 0.01))
posts_top_1 = posts_por_usuario.head(top_1_perc_qty).sum()
pct_posts_top_1 = (posts_top_1 / len(df_core)) * 100
print(f"El {pct_posts_top_1:.2f}% de los posts fueron generados por el top 1% de usuarios activos.")

# Tendencia mensual
df_core['post_month'] = df_core['post_created_at'].dt.to_period('M')
print("\nActividad de posts por mes:")
print(df_core['post_month'].value_counts().sort_index())

--- Distribuciones ---

Usuarios unicos por plan:
plan_type
enterprise      82
free          1532
pro            362
Name: user_id, dtype: int64

Actividad (posts) por pais (Top 5):
country
US    1803
BR    1526
AR    1123
PY     799
MX     772
Name: count, dtype: int64

Actividad (posts) por categoria:
post_category
tech         1333
life         1013
sports        941
science       828
finance       824
gaming        794
music         657
education     650
health        645
travel        476
Name: count, dtype: int64

Actividad (posts) por dispositivo:
device_type
web        4099
mobile     3498
desktop     564
Name: count, dtype: int64

--- Engagement ---

Votos por plan (media y mediana):
                mean  median
plan_type                   
enterprise  7.590062     7.0
free        6.676948     6.0
pro         7.677211     7.0

Votos por categoria (Promedio):
post_category
tech         7.693923
science      7.576087
gaming       7.177582
travel       6.869748
finance      6.856

### 5️⃣ Product Decisions (Basadas en evidencia)

#### 8.1 Preguntas:
- **¿Qué segmento priorizarías y por qué?**
  El segmento *Enterprise/Pro* ya que muestran en la distribución métricas consistentemente valiosas o porque retienen más actividad sostenida. Además, enfoques de retención hacia usuarios de mobile suelen ser clave.

- **¿Qué parte del tablero "mentía" antes de limpiar?**
  La métrica `days_since_signup`. Mostraba un % importante de 'mismatches' comparado con un recálculo limpio, generando posibles errores en el tiempo de vida (LTV) del usuario.

- **¿Qué nuevo dato agregarías al tracking?**
  Tiempo promedio leyendo el post o si incluye multimedia, para que el número de votos no sea el único reflejo de valor real (`engagement puro`).

#### 8.2 Acciones:
1. **Campaña de Retención:** El Top 1% agrupa demasiados posts. Se debe incentivar al resto de la base a publicar, por ejemplo, dando un "logro/bonus" en el primer mes de uso.
2. **Mejora Mobile:** Si la categoría o equipo móvil domina, se debe priorizar recursos de ingeniería en optimizar su interfaz en próximas versiones.

In [6]:
### 6️⃣ Exportacion Final

if 'post_month' in df_core.columns:
    df_core = df_core.drop(columns=['post_month'])

df_core.to_csv('clean_product_activity.csv', index=False)
df_quarantine.to_csv('quarantine_product_activity.csv', index=False)

metrics = {
    'core_rows': total_core_dedup,
    'avg_votes_per_row': promedio_fila,
    'avg_votes_per_user': promedio_usuario,
    'quarantine_percent': pct_quarantine,
    'top_1_percent_post_share': pct_posts_top_1
}
pd.DataFrame([metrics]).to_csv('metrics_summary.csv', index=False)

print("\n¡Challenge completado con exito! Archivos exportados: clean_product_activity.csv, quarantine_product_activity.csv, metrics_summary.csv")


¡Challenge completado con exito! Archivos exportados: clean_product_activity.csv, quarantine_product_activity.csv, metrics_summary.csv
